# Upload Dataset to Vayu Object Storage

In [11]:
import os
import boto3

from pathlib import Path

from dotenv import load_dotenv

_repo_root = Path.cwd()
if not (_repo_root / "requirements.txt").is_file():
    _repo_root = _repo_root.parent
load_dotenv(_repo_root / ".env", override=True)

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"


def _require_env(name: str) -> str:
    """Return a required env var or abort if missing or still a placeholder."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise EnvironmentError(
            f"Missing required environment variable {name!r}. "
            "Copy .env.example to .env at the ask-it repo root and set a real value "
            "(see 01_dataset/README.md)."
        )
    if "<" in value or "***" in value:
        raise EnvironmentError(
            f"Environment variable {name!r} looks like a placeholder. "
            "Update ask-it/.env with real credentials."
        )
    return value


# Credentials and bucket from environment — never paste secrets into notebook cells
AWS_KEY = _require_env("VAYU_S3_KEY")
AWS_SECRET = _require_env("VAYU_S3_SECRET")
S3_ENDPOINT = _require_env("VAYU_S3_ENDPOINT")
BUCKET = _require_env("VAYU_S3_BUCKET")

# Local folder path to upload
LOCAL_FOLDER = "docs"  # Change this to your dataset folder

# S3 folder/destination path (target root folder in bucket)
S3_PREFIX = "ask-it/docs/"  # S3 'folder' (adds trailing slash)

# Create S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    endpoint_url=S3_ENDPOINT
)

def upload_folder(local_folder, bucket, s3_prefix):
    for root, _, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")
            print(f"Uploading {local_path} → s3://{bucket}/{s3_key}")
            s3.upload_file(local_path, bucket, s3_key)
    print("DONE: Folder uploaded successfully")

upload_folder(LOCAL_FOLDER, BUCKET, S3_PREFIX)

Uploading docs/OG-PMKMY(Kisan Maan Dhan)_ENG.pdf → s3://thebrahmandgurus-bucket/ask-it/docs/OG-PMKMY(Kisan Maan Dhan)_ENG.pdf
Uploading docs/Operational_Guidelines_of_Namo_Drone_Didi_Scheme.pdf → s3://thebrahmandgurus-bucket/ask-it/docs/Operational_Guidelines_of_Namo_Drone_Didi_Scheme.pdf
Uploading docs/markdown-example.md → s3://thebrahmandgurus-bucket/ask-it/docs/markdown-example.md
Uploading docs/R B I Master Circular on DAY _ NRLM 2024-25.pdf → s3://thebrahmandgurus-bucket/ask-it/docs/R B I Master Circular on DAY _ NRLM 2024-25.pdf
Uploading docs/text-example.txt → s3://thebrahmandgurus-bucket/ask-it/docs/text-example.txt
Uploading docs/KUSUM 2024.pdf → s3://thebrahmandgurus-bucket/ask-it/docs/KUSUM 2024.pdf
Uploading docs/RevisedPM-KISANOperationalGuidelines(English).pdf → s3://thebrahmandgurus-bucket/ask-it/docs/RevisedPM-KISANOperationalGuidelines(English).pdf
Uploading docs/KCC-RURAL BANKS.pdf → s3://thebrahmandgurus-bucket/ask-it/docs/KCC-RURAL BANKS.pdf
Uploading docs/Kisan C

# Download Dataset from Vayu Object Storage

In [12]:
import os
import boto3
from botocore.config import Config

# 1. Reset the checksum environment variables to standard accepted defaults
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

# 2. Configure the client for alternative object stores
s3_config = Config(
    s3={'addressing_style': 'path'},
    signature_version='s3v4'
)

# 3. Securely initialize the client
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    endpoint_url=S3_ENDPOINT,
    config=s3_config  
)

def download_folder(bucket, s3_prefix, local_folder):
    paginator = s3.get_paginator('list_objects_v2')
    
    for page in paginator.paginate(Bucket=bucket, Prefix=s3_prefix):
        if "Contents" not in page:
            print("No objects found in S3 folder!")
            return
        for obj in page["Contents"]:
            key = obj["Key"]
            
            if key.endswith('/'):
                continue
            
            rel_path = os.path.relpath(key, s3_prefix)
            local_path = os.path.join(local_folder, rel_path)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            
            print(f"Streaming s3://{bucket}/{key} → {local_path}")
            
            # 4. Stream the raw byte data cleanly to local storage
            # This completely bypasses the multipart conditional ETag gates
            response = s3.get_object(Bucket=bucket, Key=key)
            with open(local_path, 'wb') as f:
                f.write(response['Body'].read())
                
    print("DONE: Folder downloaded successfully")

# Execute the process
download_folder(BUCKET, S3_PREFIX, LOCAL_FOLDER)

Streaming s3://thebrahmandgurus-bucket/ask-it/docs/FINALSchemeGuidelinesAIF.pdf → docs/FINALSchemeGuidelinesAIF.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/FPO_Scheme_Guidelines_FINAL_English.pdf → docs/FPO_Scheme_Guidelines_FINAL_English.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/Fasal Bima Yojana.pdf → docs/Fasal Bima Yojana.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/KCC-RURAL BANKS.pdf → docs/KCC-RURAL BANKS.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/KCC-RURAL COOPERATIVE BANKS.pdf → docs/KCC-RURAL COOPERATIVE BANKS.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/KUSUM 2024.pdf → docs/KUSUM 2024.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/Kisan Credit Card.pdf → docs/Kisan Credit Card.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/OG-PMKMY(Kisan Maan Dhan)_ENG.pdf → docs/OG-PMKMY(Kisan Maan Dhan)_ENG.pdf
Streaming s3://thebrahmandgurus-bucket/ask-it/docs/Operational_Guidelines_of_Namo_Drone_Didi_Scheme.pdf → do